# LangChain Output Parser Base Reference

Developer-facing statements defined in `langchain_core.output_parsers.base`.

# `OutputParserLike`

Runnable type alias for a language-model output parser.

```python
OutputParserLike = Runnable[LanguageModelOutput, T]
```

# `BaseLLMOutputParser: ABC, Generic[T]`

Abstract base class for parsing model generations into structured output.

## Required subclass hooks

### `parse_result`

Parses candidate generations for one model input.

```python
@abstractmethod
parse_result(
    self,
    result: list[Generation], # Candidate generations for a single model input
    *,
    partial: bool = False, # Whether the supplied generations represent a partial result
) -> T # Structured parsed output
```

A concrete subclass must implement this method. The abstract method body does not explicitly raise `NotImplementedError`.

## Methods

### `aparse_result`

Runs `parse_result()` asynchronously through `run_in_executor()`.

```python
async aparse_result(
    self,
    result: list[Generation], # Candidate generations for a single model input
    *,
    partial: bool = False, # Whether the supplied generations represent a partial result
) -> T # Structured parsed output

In [ ]:
from langchain_core.output_parsers.base import BaseLLMOutputParser # Import the abstract parser
from langchain_core.outputs import Generation # Import the Generation class


class FirstGenerationParser(BaseLLMOutputParser[str]): # Create a concrete output parser
    def parse_result( # Implement the required parsing method
        self,
        result: list[Generation], # Receive candidate generations
        *,
        partial: bool = False, # Indicate whether the result is partial
    ) -> str:
        if not result: # Check whether the generation list is empty
            raise ValueError("No generations were provided.") # Raise an error for missing output

        text = result[0].text.strip() # Read and clean the first generation

        if partial: # Check whether this is a partial result
            return f"Partial result: {text}" # Return partial parsed output

        return f"Final result: {text}" # Return final parsed output


parser = FirstGenerationParser() # Create the concrete parser

generations = [ # Create candidate model generations
    Generation(text="  Python is easy to learn.  "), # Create the first candidate
    Generation(text="Python is widely used."), # Create the second candidate
] # Finish the generation list

sync_result = parser.parse_result(generations) # Parse synchronously
print(sync_result) # Display the synchronous result

partial_result = parser.parse_result( # Parse the output as a partial result
    generations,
    partial=True,
)
print(partial_result) # Display the partial result

async_result = await parser.aparse_result(generations) # Parse asynchronously in Jupyter
print(async_result) # Display the asynchronous result

# `BaseGenerationOutputParser: BaseLLMOutputParser[T], RunnableSerializable[LanguageModelOutput, T]`

Abstract runnable base class for parsers that operate directly on model generations.

A concrete subclass must implement the inherited `parse_result()` method.

## Properties

### `InputType`

Returns the accepted runnable input type.

```python
InputType: Any
```

The value is `str | AnyMessage`.

### `OutputType`

Returns the parser output type used for runnable schema construction.

```python
OutputType: type[T]
```

The implementation returns the generic type variable `T` cast to `type[T]`.

## Methods

### `invoke`

Synchronously parses a string or message input through the runnable callback infrastructure.

```python
@override
invoke(
    self,
    input: str | BaseMessage, # String or message to parse
    config: RunnableConfig | None = None, # Optional runnable configuration
    **kwargs: Any, # Additional accepted arguments
) -> T # Structured parsed output
```

A `BaseMessage` is wrapped in a single `ChatGeneration`; a string is wrapped in a single `Generation`. The resulting list is passed to `parse_result()`. The runnable call uses run type `"parser"`.

### `ainvoke`

Asynchronously parses a string or message input through the runnable callback infrastructure.

```python
@override
async ainvoke(
    self,
    input: str | BaseMessage, # String or message to parse
    config: RunnableConfig | None = None, # Optional runnable configuration
    **kwargs: Any | None, # Additional accepted arguments
) -> T # Structured parsed output
```

A `BaseMessage` is wrapped in a single `ChatGeneration`; a string is wrapped in a single `Generation`. The resulting list is passed to `aparse_result()`. The runnable call uses run type `"parser"`.

In [ ]:
from typing import Any # Import Any for the parsed output dictionary

from langchain_core.messages import AIMessage # Import a real LangChain message
from langchain_core.output_parsers.base import BaseGenerationOutputParser # Import the abstract parser
from langchain_core.outputs import ChatGeneration, Generation # Import generation classes


class GenerationDetailsParser(BaseGenerationOutputParser[dict[str, Any]]): # Create a concrete parser
    def parse_result( # Implement the required parsing method
        self,
        result: list[Generation], # Receive candidate generations
        *,
        partial: bool = False, # Indicate whether the result is incomplete
    ) -> dict[str, Any]:
        if not result: # Check whether a generation was provided
            raise ValueError("No generations were provided.") # Reject an empty list

        first_generation = result[0] # Select the first candidate generation

        return { # Return structured generation details
            "text": first_generation.text.strip(), # Return cleaned text
            "generation_type": type(first_generation).__name__, # Return the generation class
            "is_chat_generation": isinstance(first_generation, ChatGeneration), # Check its type
            "partial": partial, # Return the partial flag
        }


parser = GenerationDetailsParser() # Create the concrete parser

string_result = parser.invoke( # Parse a normal string
    "  Python is easy to learn.  ", # Provide string output
    config={"run_name": "parse_string"}, # Name the parser run
)

print("String result:", string_result) # Display the string result

message = AIMessage( # Create a structured chat message
    content="LangChain supports structured messages."
)

message_result = parser.invoke( # Parse the AIMessage
    message, # Provide the message
    config={"run_name": "parse_message"}, # Name the parser run
)

print("Message result:", message_result) # Display the message result

async_result = await parser.ainvoke( # Parse asynchronously in Jupyter
    "  Asynchronous parsing also works.  ", # Provide string output
    config={"run_name": "parse_async"}, # Name the parser run
)

print("Async result:", async_result) # Display the asynchronous result
print("Accepted input type:", parser.InputType) # Display the accepted input type
print("Parser output type:", parser.OutputType) # Display the generic output type

# `BaseOutputParser: BaseLLMOutputParser[T], RunnableSerializable[LanguageModelOutput, T]`

Abstract runnable base class for parsers that convert a single text output into structured data.

A concrete subclass must implement `parse()`.

## Properties

### `InputType`

Returns the accepted runnable input type.

```python
InputType: Any
```

The value is `str | AnyMessage`.

### `OutputType`

Infers the parser output type from the first Pydantic generic type argument found in the class hierarchy.

```python
OutputType: type[T]
```

Raises `TypeError` when no output type can be inferred. A subclass can override this property to provide the type explicitly.

## Required subclass hooks

### `parse`

Parses one model-output string.

```python
@abstractmethod
parse(
    self,
    text: str, # Model output to parse
) -> T # Structured parsed output
```

The abstract method body does not explicitly raise `NotImplementedError`.

## Optional subclass hooks

### `get_format_instructions`

Returns instructions describing the expected model-output format.

```python
get_format_instructions(
    self,
) -> str # Output-format instructions
```

The default implementation raises `NotImplementedError`.

## Methods

### `invoke`

Synchronously parses a string or message through the runnable callback infrastructure.

```python
@override
invoke(
    self,
    input: str | BaseMessage, # String or message to parse
    config: RunnableConfig | None = None, # Optional runnable configuration
    **kwargs: Any, # Additional accepted arguments
) -> T # Structured parsed output
```

A `BaseMessage` is wrapped in a single `ChatGeneration`; a string is wrapped in a single `Generation`. The resulting list is passed to `parse_result()`. The runnable call uses run type `"parser"`.

### `ainvoke`

Asynchronously parses a string or message through the runnable callback infrastructure.

```python
@override
async ainvoke(
    self,
    input: str | BaseMessage, # String or message to parse
    config: RunnableConfig | None = None, # Optional runnable configuration
    **kwargs: Any | None, # Additional accepted arguments
) -> T # Structured parsed output
```

A `BaseMessage` is wrapped in a single `ChatGeneration`; a string is wrapped in a single `Generation`. The resulting list is passed to `aparse_result()`. The runnable call uses run type `"parser"`.

### `parse_result`

Parses only the first candidate generation.

```python
@override
parse_result(
    self,
    result: list[Generation], # Candidate generations ordered by likelihood
    *,
    partial: bool = False, # Accepted for parser compatibility
) -> T # Structured parsed output
```

The method calls `parse(result[0].text)`. The `partial` value is not forwarded to `parse()`.

### `aparse_result`

Runs `parse_result()` asynchronously through `run_in_executor()`.

```python
async aparse_result(
    self,
    result: list[Generation], # Candidate generations ordered by likelihood
    *,
    partial: bool = False, # Whether the supplied generations represent a partial result
) -> T # Structured parsed output
```

### `aparse`

Runs `parse()` asynchronously through `run_in_executor()`.

```python
async aparse(
    self,
    text: str, # Model output to parse
) -> T # Structured parsed output
```

### `parse_with_prompt`

Parses a completion while accepting the originating prompt as optional context.

```python
parse_with_prompt(
    self,
    completion: str, # Model output to parse
    prompt: PromptValue, # Prompt that produced the completion
) -> Any # Structured parsed output
```

The default implementation ignores `prompt` and delegates to `parse(completion)`.

### `dict`

Deprecated alias for `asdict()`.

```python
@deprecated("1.4.2", alternative="asdict", removal="2.0.0")
@override
dict(
    self,
    **kwargs: Any, # Arguments forwarded to asdict()
) -> builtins.dict[str, Any] # Dictionary representation
```

### `asdict`

Returns the Pydantic model representation of the parser.

```python
asdict(
    self,
    **kwargs: Any, # Arguments forwarded to model_dump()
) -> builtins.dict[str, Any] # Dictionary representation
```

The method calls `model_dump()` and adds an `"_type"` entry when the subclass implements the private serialization-type property. A `NotImplementedError` from that property is suppressed.

In [ ]:
from langchain_core.messages import AIMessage # Import a real LangChain message
from langchain_core.output_parsers import BaseOutputParser # Import the abstract output parser
from langchain_core.outputs import Generation # Import the Generation class
from langchain_core.prompts import PromptTemplate # Import PromptTemplate for creating a PromptValue


class CommaSeparatedIntegerParser(BaseOutputParser[list[int]]): # Create a concrete output parser
    separator: str = "," # Store the separator used between values

    @property
    def _type(self) -> str: # Define the parser type used by asdict
        return "comma_separated_integer_parser" # Return the parser type name

    def parse(self, text: str) -> list[int]: # Implement the required parsing method
        values = text.strip().split(self.separator) # Split the text into separate values
        return [int(value.strip()) for value in values] # Convert each value to an integer

    def get_format_instructions(self) -> str: # Describe the expected output format
        return "Return integers separated by commas, for example: 10, 20, 30." # Return instructions


parser = CommaSeparatedIntegerParser() # Create the parser

print("Format instructions:", parser.get_format_instructions()) # Display format instructions
print("Input type:", parser.InputType) # Display the accepted input type
print("Output type:", parser.OutputType) # Display the inferred output type

direct_result = parser.parse("10, 20, 30") # Call parse directly
print("\nDirect parse:", direct_result) # Display the result

string_result = parser.invoke( # Parse a string using the runnable interface
    "1, 2, 3", # Provide model-output text
    config={"run_name": "parse_integer_string"}, # Name the parser run
)

print("String invoke:", string_result) # Display the parsed string

message = AIMessage(content="4, 5, 6") # Create an AI message
message_result = parser.invoke(message) # Parse the message

print("Message invoke:", message_result) # Display the parsed message

generations = [ # Create multiple candidate generations
    Generation(text="7, 8, 9"), # Create the first candidate
    Generation(text="100, 200"), # Create the second candidate
]

generation_result = parser.parse_result(generations) # Parse only the first generation
print("Generation result:", generation_result) # Display the result

async_invoke_result = await parser.ainvoke("11, 12, 13") # Invoke asynchronously in Jupyter
print("Async invoke:", async_invoke_result) # Display the result

async_parse_result = await parser.aparse("14, 15, 16") # Call parse asynchronously
print("Async parse:", async_parse_result) # Display the result

async_generation_result = await parser.aparse_result(generations) # Parse generations asynchronously
print("Async generation result:", async_generation_result) # Display the result

prompt = PromptTemplate.from_template( # Create a prompt template
    "Return three integers related to {topic}."
).format_prompt(topic="Python") # Convert it into a PromptValue

prompt_result = parser.parse_with_prompt( # Parse with the original prompt
    "17, 18, 19", # Provide the completion
    prompt, # Provide the PromptValue
)

print("Parse with prompt:", prompt_result) # Display the result
print("Parser dictionary:", parser.asdict()) # Display the parser representation